# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIRˆ² dataset using the `mlcroissant` library with detailed step-by-step instructions, referencing all Croissant entities using their `@id` identifiers as per best practice.

### Dataset Source
This notebook loads the dataset directly from its FAIR-compliant Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed.
!pip install -q mlcroissant

## 1. Data Loading
Load the FAIRˆ² dataset: metadata, schema, and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object, and print its title and description
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
List available record sets, fields, and their `@id`s. All references use the unique `@id` for each entity as defined by Croissant schema best practices:

In [ ]:
from pprint import pprint

print("\nAvailable record sets and their fields (referenced by @id):\n")

# Record set objects (list of mlcroissant.entities.record_set.RecordSet)
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("")

Below, we preview the records of the main record set (id referenced by its `@id`).

In [ ]:
# Print a sample from each record set using their @id.

# We'll find the main data record set by heuristics: look for record set with the most fields or the most relevant name

preview_count = 2

for rs in record_sets:
    print(f"\nSample records for RecordSet '{rs.name}' (@id: {rs.id}):")
    records = dataset.records(record_set=rs.id)
    for i, rec in enumerate(records):
        if i >= preview_count:
            break
        pprint(rec)


## 3. Data Extraction
Load data from the main record set(s) into pandas DataFrames for subsequent analysis. 

Again, all entities are referenced using their Croissant `@id` fields.

In [ ]:
# Build a list of all record set @id's
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # The records iterator yields dicts with field @ids as keys
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for record set @id: {record_set_id} with shape {dataframes[record_set_id].shape}")
        print(f"Columns (@id): {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head(3)) # Preview
    else:
        print(f"No records found for record set @id: {record_set_id}")


## 4. Exploratory Data Analysis (EDA)

Let's perform some typical analysis steps on the main record set, using column `@id`s for all references. 

Typical EDA: filter records based on a numeric field, normalize that field, group by a categorical attribute, and check for outliers.

**Note:** We select a numeric field and a group field by their `@id` below. You may adjust to any available `@id` as needed.

In [ ]:
import numpy as np

# Choose main data record set. If only one, take that; else, pick first with records.
main_record_set_id = None
for rid in dataframes:
    if not dataframes[rid].empty:
        main_record_set_id = rid
        break

if main_record_set_id is None:
    raise RuntimeError("No non-empty record sets found.")

main_df = dataframes[main_record_set_id]

# Use the record set object to print field ids for reference
for rs in record_sets:
    if rs.id == main_record_set_id:
        print(f"\nField names and @ids for main record set '{rs.name}':")
        for field in rs.fields:
            print(f"  - Name: {field.name:40s}  @id: {field.id:50s}   dataType: {field.data_type}")
        break

# Choose a numeric field and a group/categorical field by their @ids
# Below are example heuristics. Adjust if needed.
# We'll try 'schema:age' for numeric, or fallback to any Integer/Float

numeric_field_id = None
group_field_id = None
for rs in record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if field.data_type in ['Integer', 'Float', 'Number'] and (numeric_field_id is None):
                numeric_field_id = field.id
            if field.data_type in ['Text', 'String', 'Boolean'] and (group_field_id is None):
                group_field_id = field.id
        break

if numeric_field_id is None:
    raise ValueError("No numeric field found in main record set.")

print(f"\nNumeric field selected: {numeric_field_id}")
print(f"Group (categorical) field selected: {group_field_id}")

# Make sure the selected columns exist
if numeric_field_id not in main_df.columns:
    raise ValueError(f"Numeric field '@id' {numeric_field_id} not found in DataFrame columns.")
if group_field_id not in main_df.columns:
    group_field_id = None  # Not required for rest of EDA

# Convert numeric field to numeric if needed
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filter out obvious outliers (if any), e.g., keep values within 1st and 99th percentile
low, high = main_df[numeric_field_id].quantile([0.01, 0.99])
eda_df = main_df[(main_df[numeric_field_id] >= low) & (main_df[numeric_field_id] <= high)]

threshold = eda_df[numeric_field_id].mean()

# Filter records above threshold
filtered_df = eda_df[eda_df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized values for {numeric_field_id} (first rows):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field (if categorical field available)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
    print(f"\nGrouped means of {numeric_field_id} by {group_field_id}:\n")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field (by `@id`) and, if available, group by the selected categorical field (also by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(eda_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.tight_layout()
plt.show()

# Optional: Boxplot by group_field (if exists)
if group_field_id and group_field_id in eda_df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=eda_df)
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrated loading a FAIR-compliant biomedical dataset defined by a Croissant schema, exploring its record sets and fields by unique `@id`, and performing basic processing and analysis. All data manipulations and visualizations referenced dataset attributes using their Croissant `@id` as required for robust, schema-aware analysis workflows.

You can further extend this notebook to perform detailed domain-specific analyses, or to prepare the data for machine learning experiments leveraging the clarity and portability of the Croissant metadata model.
